# Creating the final Dataset

Each football match should have 4 more features: <b>HomeStartingTeamRating</b>, <b>AwayStartingTeamRating</b>, <b>HomeSubTeamRating</b>, and <b>AwaySubTeamRating</b>. <br>These features are calculated based on the player dataset, each player have a Rating and other stats that should allow a more precise prediction

## Create New Features
1. Read player dataset and save player IDs
2. Extract team lineup and substitutes from match JSONs and add them to an array with "Name_ID" format
3. If player ID isn't within the ones of point 1, add "Name,ID" to a log array

In [41]:
import os
import json
import pandas as pd
import time
from datetime import datetime
import glob
import openpyxl
import requests
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import unicodedata
from bs4 import BeautifulSoup

In [42]:
#1. Read player dataset and save player IDs
percorso_csv = r".\Datasets\Styled_STATS_giocatori_serie_A.xlsx"
df_players = pd.read_excel(percorso_csv)

giocatori_noti = set(df_players['ID Giocatore'].astype(str))

In [43]:
# 2. Extract team lineup and substitutes from match JSONs and add them to an array with "Name_ID" format

cartelle_json = [
    r".\Datasets\MatchStatsJSON_2324",
    r".\Datasets\MatchStatsJSON_2425",
    r".\Datasets\MatchStatsJSON_2526"
]

lineup_partite = []
log_mancanti = set() # set per evitare duplicati nel log

In [44]:
# Funzione per estrarre i giocatori e inserirli in un array
def estrai_giocatori(team_data, player_type):
    array_giocatori = []
    for player in team_data.get(player_type, []):
        
        # Se analizzo la panchina, verifico che sia entrato andando in performance -> substitutionEvents e cercando un evento type = "subIn"
        if player_type == "subs":
            performance = player.get("performance", {})
            sub_events = performance.get("substitutionEvents", []) if performance else []
            
            # Verifichiamo se c'è almeno un evento di tipo "subIn"
            ha_giocato = any(event.get("type") == "subIn" for event in sub_events)

            # Se non è mai entrato, lo saltiamo e passiamo al prossimo giocatore
            if not ha_giocato:
                continue 

        player_id = str(player.get("id"))
        player_name = player.get("name")
        
        # aggiungo ad array
        array_giocatori.append(f"{player_name}_{player_id}")
        
        # se non ho gia player lo metto nei log da printarte
        if player_id not in giocatori_noti:
            log_mancanti.add(f"{player_name},{player_id}")
            
    return ", ".join(array_giocatori) #ritorno una string (array non supportato per excel)

In [45]:
# Itera su tutte le cartelle e i relativi file JSON
for cartella in cartelle_json:
    JSONs = os.path.join(cartella, "*.json")
    for filepath in glob.glob(JSONs):
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            try:
                # nel json la formazione è in content -> lineup -> homeTeam/awayTeam -> starters/subs
                lineup = data["content"]["lineup"]
                home_team = lineup["homeTeam"]
                away_team = lineup["awayTeam"]
                
                home_starters = estrai_giocatori(home_team, "starters")
                home_subs = estrai_giocatori(home_team, "subs")
                away_starters = estrai_giocatori(away_team, "starters")
                away_subs = estrai_giocatori(away_team, "subs")
                
                # estraggo anche matchID per il merge con dataset completo
                match_id = data["general"]["matchId"]
                
                lineup_partite.append({
                    "ID Partita": match_id,
                    "HomeStartingTeam": home_starters,
                    "AwayStartingTeam": away_starters,
                    "HomeSubPlayers": home_subs,
                    "AwaySubPlayers": away_subs
                })
            except KeyError as e:
                print(f"Errore in {filepath} (chiave {e} mancante)")

In [46]:
# Genera il DataFrame per le partite che puoi facilmente unire con 'SerieA_Fixture_Stats'
df_nuove_colonne = pd.DataFrame(lineup_partite)
print("Estrazione completata. Prime righe del risultato:")
print(df_nuove_colonne.head())

Estrazione completata. Prime righe del risultato:
  ID Partita                                   HomeStartingTeam  \
0    4230622  Josep Martinez_772168, Radu Dragusin_1203661, ...   
1    4230627  Andrea Consigli_39170, Jeremy Toljan_443966, M...   
2    4230624  Wladimiro Falcone_362200, Valentin Gendrey_107...   
3    4230629  Wojciech Szczesny_169718, Federico Gatti_11892...   
4    4230621  Simone Scuffet_423223, Nahitan Nández_643573, ...   

                                    AwayStartingTeam  \
0  Guillermo Ochoa_31047, Matteo Lovato_1047570, ...   
1  Lukasz Skorupski_178732, Lorenzo De Silvestri_...   
2  Vanja Milinkovic-Savic_543021, Mergim Vojvoda_...   
3  Lorenzo Montipo_491784, Giangiacomo Magnani_83...   
4  Stefano Turati_869054, Pol Lirola_769335, Ilar...   

                                      HomeSubPlayers  \
0  Caleb Ekuban_424614, Koni De Winter_1200630, B...   
1  Uros Racic_764624, Cristian Volpato_1308909, E...   
2  Rémi Oudin_780491, Gabriel Strefezza_83

In [47]:

# Converti il set in una lista (l'array 'log' da te richiesto)
log = list(log_mancanti)

print(f"\nSono stati trovati {len(log)} giocatori mancanti")
for entry in log:
    print(entry)



Sono stati trovati 506 giocatori mancanti
Berkan Kutlu,1162410
Theo Hernández,724371
Giacomo Bonaventura,127460
Luis Muriel,196484
Silvan Hefti,678384
Joan Gonzalez,1289600
Tommaso Martinelli,1420260
Matias Vina,840184
Cheick Conde,1132551
Balthazar Pierret,1125446
Tete Morente,801213
Chaka Traorè,1224408
Nikola Milenkovic,766611
Jose Machin,671621
Emanuel Vignato,814138
Juan Musso,389050
Christian Kouamé,776940
Ionut Andrei Radu,681213
Joao Ferreira,1072257
Nedim Bajrami,822794
Elayis Tavsan,983047
Matteo Guendouzi,800609
Axel Guessand,1292195
Mattia Viti,1116749
Ismael Bennacer,609624
Valentin Gendrey,1077653
Laurs Skjellerup,1163273
Alessandro Vogliacco,958401
Simy,281993
Simon Kjær,35112
Joel Pohjanpalo,290604
Aaron Ciammaglichella,1436505
Rade Krunic,438456
Edoardo Bove,1212866
Yannik Engelhardt,1269497
Daniele Baselli,195413
Luca Garritano,362119
Patrick Ciurria,480021
Beto,1079485
Tyronne Ebuehi,561844
Alessandro Gabrielloni,566776
Dachi Lordkipanidze,1668225
Ciro Immobile,1616

In [48]:
dati_log = [entry.split(',') for entry in log]

df_log = pd.DataFrame(dati_log, columns=['Nome Giocatore', 'ID Giocatore'])

percorso_excel_log = r'.\Datasets\log_giocatori_mancanti.xlsx'
df_log.to_excel(percorso_excel_log, index=False)

In [49]:
# Merge con dataset completo
df_main = pd.read_excel(r".\Datasets\Styled_SerieA_Fixture_Stats_Total.xlsx")
df_main['ID Partita'] = df_main['ID Partita'].astype(str)
df_nuove_colonne['ID Partita'] = df_nuove_colonne['ID Partita'].astype(str)
df_merged = pd.merge(df_main, df_nuove_colonne, on='ID Partita', how='left')

#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Risultati e Statistiche"
ws_data.views.sheetView[0].showGridLines = True


# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)


# Populate Data Sheet
headers = list(df_merged.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df_merged.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    # Apply link to URL FotMob column
    url_val = r_data["URL FotMob"]
    if url_val:
        cell = ws_data.cell(row=row_num, column=headers.index("URL FotMob") + 1)
        cell.hyperlink = url_val
        cell.font = Font(name="Segoe UI", size=10, color="0000FF", underline="single")
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            if col_idx != headers.index("URL FotMob") + 1:
                c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]: # Numeric or short text fields
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)


ws_data.freeze_panes = "A2"
path_stats_styled = r".\Datasets\Styled_Fixtures_Lineup.xlsx"
wb.save(path_stats_styled)

### Download of Additional Players Stats
The code is the same used for the 2025/2026 players

In [50]:
def clean_header_string(text):
    if not text:
        return ""
    text = text.replace('\u2026', '...')
    return text.encode('latin-1', 'ignore').decode('latin-1')

In [51]:
def clear_player_name(text):
    if not text:
        return ""
    
    # tutto in minuscolo
    text_clean = str(text).lower().strip()
    
    # spazi con il trattino
    text_clean = text_clean.replace(" ", "-")
    
    # tolgo i caratteri accentati. Es: "à" diventa "a" + "accen_grave"
    text_normalized = unicodedata.normalize('NFD', text_clean)
    
    # tenieamo la stringa solo con i caratteri ASCII base
    slug = "".join(
        c for c in text_normalized 
        if (c.isalnum() and unicodedata.category(c) != 'Mn') or c in ['_', '-']
    )
    
    # tolgo trattini doppi
    while '--' in slug:
        slug = slug.replace('--', '-')
        
    return slug.strip('-')

In [52]:
input_excel_path = r'.\Datasets\log_giocatori_mancanti.xlsx'

In [53]:
try:
    df = pd.read_excel(input_excel_path, engine='openpyxl')
    
    print(f"File caricato con successo! Trovati {len(df)} giocatori.\n")

except FileNotFoundError:
    print(f"\033[91m[Errore] Impossibile trovare il file '{input_excel_path}'. Verifica il percorso.\033[0m")
except Exception as e:
    print(f"\033[91m[Errore] Si è verificato un problema durante la lettura del file: {e}\033[0m")

File caricato con successo! Trovati 506 giocatori.



In [54]:
#test
clear_player_name("João Mário")

'joao-mario'

In [55]:
cookie_sporco = r"""g_state={"i_l":4,"i_ll":1781516391377,"i_b":"bXYS9zdTWGlM9fWGhj9cOE9j3P6ctVHcG87fWVXrJv8","i_e":{"enable_itp_optimization":0},"i_et":1779710661468,"i_p":1783932355386}; _hjSessionUser_2585474=eyJpZCI6ImNlMmRkOTllLWE5NDMtNWFjYS05YzE3LTI2MTQxNTdhMTRlOSIsImNyZWF0ZWQiOjE3Nzk3MTA2NjE2MzQsImV4aXN0aW5nIjp0cnVlfQ==; FCCDCF=%5Bnull%2Cnull%2Cnull%2C%5B%22CQkw6oAQkw6oAEsACBITCgFoAP_gAEPgABpwLItB_C7dbWFDyLZ3absEeIhHx_hjasQwAAbAAmAFTBqQoJQGwmEYNAyCtCACGAAAKmTBIQIkCAAAAUAAIIAFIADMAAyQIBAIICBAiAEJAEBACEACCoAgUQCIgIJEFEQAm…FRome%22%7D; panoramaId_expiry=1781599547840; panoramaId=cb1bea02cf06f766bae9ed432fb6a9fb927a241769f4e99a50bf865f6052ae5a; panoramaIdType=panoDevice; _hjSession_2585474=eyJpZCI6ImI0Zjg5YmYxLTRiNDgtNDY3NS1hM2FhLWE3ZTdlZDc1ZDkwYiIsImMiOjE3ODE1MTYzOTA2NjgsInMiOjAsInIiOjAsInNiIjowLCJzciI6MCwic2UiOjAsImZzIjowLCJzcCI6MH0=; FCNEC=%5B%5B%22AKsRol-PuQZMOL8qecnAqh6EW0BhBDootiQfMlUvoC4saag-dMb-H2_IEtfcqKgCU8l88HP9Gfuz4pZf-s5slIA3Lel2Sleq-5_wKV3aEEYnqZNW14suEXcBYmfCyLIx9gILCUS2rOthEY78DqtzUG3AGOVaYXBMtg%3D%3D%22%5D%5D"""
cookie_pulito = clean_header_string(cookie_sporco)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Alt-Used": "www.fotmob.com",
    "Cookie": cookie_pulito
}

In [56]:
def extractNewBuildID():

    try:
        # faccio req alla home del sito
        response_home = requests.get("https://www.fotmob.com/it", headers=headers)
        soup = BeautifulSoup(response_home.text, 'html.parser')
        
        # Next.js nasconde i dati di configurazione nel tag script "__NEXT_DATA__"
        script_tag = soup.find('script', id='__NEXT_DATA__')
        
        if script_tag:
            next_data = json.loads(script_tag.string)
            build_id = next_data.get('buildId')
            print(f"\033[92mBuild ID trovato con successo: {build_id}\033[0m\n")
        else:
            raise ValueError("Tag __NEXT_DATA__ non trovato nella homepage.")
            
    except Exception as e:
        print(f"\033[91mImpossibile recuperare il Build ID. Errore: {e}\033[0m")
        exit()
    
    return build_id

build_id = extractNewBuildID()

Build ID trovato con successo: -wrCWO5C7BvHveZE0uJO0



In [57]:
print("Inizio download JSON giocatori:\n")

log = []

for index, riga in df.iterrows():
    
    id_giocatore = riga['ID Giocatore']
    nome_giocatore = riga['Nome Giocatore']
    
    clear_name = clear_player_name(nome_giocatore)
    
    file_name = f".\\Datasets\\PlayerStats\\PlayersJSONs_Additional_old\\{clear_name}_{id_giocatore}_player.json"
    
    max_tentativi = 3
    tentativo_attuale = 0
    scaricato = False      
    
    # URL dell'API ufficiale/pulita di FotMob per la squadra
    
    while tentativo_attuale < max_tentativi and not scaricato:

        url = f"https://www.fotmob.com/_next/data/{build_id}/it/players/{id_giocatore}/{clear_name}.json?lng=it&id={id_giocatore}&slug={clear_name}"
    
        try:
            response = requests.get(url, headers=headers)

            if response.status_code == 200:
                match_json = response.json()
                
                # Salvataggio del file JSON locale
                with open(file_name, "w", encoding="utf-8") as out_file:
                    import json
                    json.dump(match_json, out_file, ensure_ascii=False, indent=4)
                    
                print(f"[{index+1}/{len(df)}] Scaricato: {nome_giocatore}")
                scaricato = True

            elif response.status_code == 404:
                tentativo_attuale += 1
                print(f"\033[91m[{index+1}/{len(df)}] Dettagli non trovati (404) per: {nome_giocatore}. Verifica se lo slug è esatto.\033[0m")
                build_id = extractNewBuildID()
                time.sleep(1)
            else:
                print(f"\033[91m[{index+1}/{len(df)}] Errore {response.status_code} per: {nome_giocatore}\033[0m")
        
        except Exception as e:
            print(f"\033[91mErrore di connessione per {nome_giocatore}: {e}\033[0m")

    if not scaricato:
        string_log = f"Impossibile scaricare {nome_giocatore} dopo {max_tentativi} tentativi."
        log.append(string_log)
        print(f"\033[91m-> Impossibile scaricare {nome_giocatore} dopo {max_tentativi} tentativi. Passo al prossimo.\033[0m")
        
    # Pausa di 1 secondo tra le richieste per rispettare i server ed evitare ban ip
    time.sleep(1)


Inizio download JSON giocatori:

[1/506] Scaricato: Berkan Kutlu
[2/506] Scaricato: Theo Hernández
[3/506] Scaricato: Giacomo Bonaventura
[4/506] Scaricato: Luis Muriel
[5/506] Scaricato: Silvan Hefti
[6/506] Scaricato: Joan Gonzalez
[7/506] Scaricato: Tommaso Martinelli
[8/506] Scaricato: Matias Vina
[9/506] Scaricato: Cheick Conde
[10/506] Scaricato: Balthazar Pierret
[11/506] Scaricato: Tete Morente
[12/506] Scaricato: Chaka Traorè
[13/506] Scaricato: Nikola Milenkovic
[14/506] Scaricato: Jose Machin
[15/506] Scaricato: Emanuel Vignato
[16/506] Scaricato: Juan Musso
[17/506] Scaricato: Christian Kouamé
[18/506] Scaricato: Ionut Andrei Radu
[19/506] Scaricato: Joao Ferreira
[20/506] Scaricato: Nedim Bajrami
[21/506] Scaricato: Elayis Tavsan
[22/506] Scaricato: Matteo Guendouzi
[23/506] Scaricato: Axel Guessand
[24/506] Scaricato: Mattia Viti
[25/506] Scaricato: Ismael Bennacer
[26/506] Scaricato: Valentin Gendrey
[27/506] Scaricato: Laurs Skjellerup
[28/506] Scaricato: Alessandro Vog

## New Serie A Teams
Every year, 3 teams get relegated in Serie B and 3 teams from Serie B get promoted in Serie A. <br>
The 3 newly promoted teams for the 2026/2027 season are:
1. Venezia
2. Frosinone
3. Monza

We already have some data for these teams in the Serie A Fixture Dataset. <br> 
What we are missing is the players data, given that after one season in Serie B it is obviuos that the team lineup got changed quite a bit. <br>
I have downloaded by hand the 3 teams JSONs from last season (<i>.\Datasets\PlayerStats\TeamsJSONs_additional</i>), now i will take the list of players and then I will download the JSONs of the ones that are not present in the dataset<br><br>
We won't need these players to be in the Fixture Dataset because there aren't any 2025/2026 match where they are included, also, for the 2024/2025 and 2023/2024 matches we already downloaded the missing players, so the dataset is complete.

### Read Teams JSONs

In [58]:
cartella_json = r'.\Datasets\PlayerStats\TeamsJSONs_additional'

oggi = datetime.now()

In [59]:
def calcola_eta(dob_str):
    #formato passato YYYY-MM-DD

    if not dob_str:
        return 'N/D'
    try:
        data_nascita = datetime.strptime(dob_str.strip(), '%Y-%m-%d')
        
        eta = oggi.year - data_nascita.year - ((oggi.month, oggi.day) < (data_nascita.month, data_nascita.day))
        return eta
    except Exception:
        return 'N/D'

In [ ]:
# per debugging
ROSSO = "\033[91m"
RESET = "\033[0m"

# Lista globale per raccogliere i dati di tutti i giocatori
tutti_i_giocatori = []

# prendo tutti i json della cartella
file_json_squadre = [f for f in os.listdir(cartella_json) if f.endswith('_team.json')]

print(f"Trovati {len(file_json_squadre)} file JSON da elaborare.\n")

for file_name in file_json_squadre:
    # Estrae il nome della squadra e id dal nome del file (es. "Inter_8636_team.json" -> "Inter", "8636")
    team = file_name.replace('_team.json', '').replace('-', ' ')
    team_and_id = team.rsplit('_', 1)
    nome_team = team_and_id[0]
    id_team = team_and_id[1]
    file_path = os.path.join(cartella_json, file_name)
    
    print(f"Elaborazione: {file_name}...")
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        # Navigazione: pageProps -> fallback
        page_props = data.get('pageProps', {})
        fallback = page_props.get('fallback', {})
        
        chiave_team = f"team-{id_team}"
        
        # navigazione fino a squad -> squad
        team_data = fallback.get(chiave_team, {})
        squad_root = team_data.get('squad', {})
        squad_list = squad_root.get('squad') if isinstance(squad_root, dict) else squad_root
        
        # Valuto se esiste il team
        if not squad_list or not isinstance(squad_list, list):
            print(f"\033[91m -> [Salto File] Struttura squad non valida in {file_name}\033[0m")
            continue
            
        # Ciclo sui reparti (Portieri, Difensori, ecc.)
        for reparto in squad_list:
            if not isinstance(reparto, dict):
                continue

            if reparto.get('title') == 'coach':
                continue

            giocatori = reparto.get('members', [])
            ruolo_reparto = reparto.get('title', 'N/D')
            
            for p in giocatori:
                id_giocatore = p.get('id')
                name_giocatore = p.get('name')
                
                if not id_giocatore or not name_giocatore:
                    continue
                
                
                dob = p.get('dateOfBirth', 'N/D')
                eta = calcola_eta(dob)

                role_obj = p.get('role')
                if isinstance(role_obj, dict):
                    role = role_obj.get('fallback', ruolo_reparto)
                else:
                    role = role_obj if role_obj else ruolo_reparto
                
                
                transfer_value = p.get('transferValue')
                
                
                rating = p.get('rating')
                if rating is None:
                    rating = 'N/D'
                
                stats = p.get('stats', {})
                rcards = p.get('rcards', 0)
                ycards = p.get('ycards', 0)
                
                # Costruisco riga del giocatore
                tutti_i_giocatori.append({
                    'codice team': id_team,
                    'nome team': nome_team,
                    'ID Giocatore': id_giocatore,
                    'Nome Giocatore': name_giocatore,
                    'dateOfBirth': dob if dob else 'N/D',
                    'età': eta,
                    'role': role,
                    'rating': rating,
                    'transferValue': transfer_value,
                    'Red cards': rcards,
                    'Yellow cards': ycards
                })
                
    except json.JSONDecodeError:
        print(f"{ROSSO} -> [Errore] File {file_name} corrotto o non è un JSON valido.{RESET}")
    except Exception as e:
        print(f"{ROSSO} -> [Errore] Problema imprevisto su {file_name}: {e}{RESET}")

Trovati 3 file JSON da elaborare.

Elaborazione: Frosinone_9891_team.json...
Elaborazione: Monza_6504_team.json...
Elaborazione: Venezia_7881_team.json...


### Download players JSONs (Same as before)
We download all the 97 players. Even if we already downloaded the JSON, there won't be any duplicates because:
1. The players currently in the dataset are the ones from the last Serie A season (none of these 97)
2. Even if we might have previously downloaded some of these 97 players in the folder "<i>.\Downloads\PlayerStats\PlayersJSONs_Additional_promoted</i>", we didn't have processed the JSONs yet

In [61]:
df = pd.DataFrame(tutti_i_giocatori)

In [62]:
print(df.head())

  codice team  nome team  ID Giocatore         Nome Giocatore dateOfBirth  \
0        9891  Frosinone       1329536            Eldin Lolic  2004-05-15   
1        9891  Frosinone       1289655      Lorenzo Palmisani  2004-06-12   
2        9891  Frosinone       1681814  Francesco Minicangeli  2006-01-12   
3        9891  Frosinone        415425         Matteo Pisseri  1991-11-21   
4        9891  Frosinone       1018462           Sergio Kalaj  2000-01-28   

   età      role rating  transferValue  Red cards  Yellow cards  
0   22    Keeper    N/D       616143.0          0             0  
1   22    Keeper   7.61      2412433.0          0             3  
2   20    Keeper    N/D        55000.0          0             0  
3   34    Keeper    N/D       148353.0          0             0  
4   26  Defender    N/D       233242.0          0             0  


In [63]:
print("Inizio download JSON giocatori:\n")

log = []

for index, riga in df.iterrows():
    
    id_giocatore = riga['ID Giocatore']
    nome_giocatore = riga['Nome Giocatore']
    
    clear_name = clear_player_name(nome_giocatore)
    
    file_name = f".\\Datasets\\PlayerStats\\PlayersJSONs_Additional_promoted\\{clear_name}_{id_giocatore}_player.json"
    
    max_tentativi = 3
    tentativo_attuale = 0
    scaricato = False      
    
    # URL dell'API ufficiale/pulita di FotMob per la squadra
    
    while tentativo_attuale < max_tentativi and not scaricato:

        url = f"https://www.fotmob.com/_next/data/{build_id}/it/players/{id_giocatore}/{clear_name}.json?lng=it&id={id_giocatore}&slug={clear_name}"
    
        try:
            response = requests.get(url, headers=headers)

            if response.status_code == 200:
                match_json = response.json()
                
                # Salvataggio del file JSON locale
                with open(file_name, "w", encoding="utf-8") as out_file:
                    import json
                    json.dump(match_json, out_file, ensure_ascii=False, indent=4)
                    
                print(f"[{index+1}/{len(df)}] Scaricato: {nome_giocatore}")
                scaricato = True

            elif response.status_code == 404:
                tentativo_attuale += 1
                print(f"\033[91m[{index+1}/{len(df)}] Dettagli non trovati (404) per: {nome_giocatore}. Verifica se lo slug è esatto.\033[0m")
                build_id = extractNewBuildID()
                time.sleep(1)
            else:
                print(f"\033[91m[{index+1}/{len(df)}] Errore {response.status_code} per: {nome_giocatore}\033[0m")
        
        except Exception as e:
            print(f"\033[91mErrore di connessione per {nome_giocatore}: {e}\033[0m")

    if not scaricato:
        string_log = f"Impossibile scaricare {nome_giocatore} dopo {max_tentativi} tentativi."
        log.append(string_log)
        print(f"\033[91m-> Impossibile scaricare {nome_giocatore} dopo {max_tentativi} tentativi. Passo al prossimo.\033[0m")
        
    # Pausa di 1 secondo tra le richieste per rispettare i server ed evitare ban ip
    time.sleep(1)


Inizio download JSON giocatori:

[1/97] Dettagli non trovati (404) per: Eldin Lolic. Verifica se lo slug è esatto.
Build ID trovato con successo: 9G-LgVGE_R9O5I9vSZpgf

[1/97] Scaricato: Eldin Lolic
[2/97] Scaricato: Lorenzo Palmisani
[3/97] Scaricato: Francesco Minicangeli
[4/97] Scaricato: Matteo Pisseri
[5/97] Scaricato: Sergio Kalaj
[6/97] Scaricato: Giorgio Cittadini
[7/97] Scaricato: Gabriele Calvani
[8/97] Scaricato: Riccardo Marchizza
[9/97] Scaricato: Jacopo Gelli
[10/97] Scaricato: Matteo Obleac
[11/97] Scaricato: Niccolò Corrado
[12/97] Scaricato: Anthony Oyono
[13/97] Scaricato: Jérémy Oyono
[14/97] Scaricato: Ilario Monterisi
[15/97] Scaricato: Gabriele Bracaglia
[16/97] Scaricato: Ilias Koutsoupias
[17/97] Scaricato: Francesco Gelli
[18/97] Scaricato: Giacomo Calò
[19/97] Scaricato: Matteo Cichella
[20/97] Scaricato: Filippo Grosso
[21/97] Scaricato: Abdoulie Ndow
[22/97] Scaricato: Antonio Fiori
[23/97] Scaricato: Edoardo Vergani
[24/97] Scaricato: Benjamin Lhassine Kone

### Read downloaded players JSONs

In [ ]:
folder_json = r'.\Datasets\PlayerStats\PlayersJSONs_Additional_promoted'

#carico il df con i dati dei giocatori (non ho excel questa volta)
df_excel = df

# trasformo la stringa "N/D" o altri valori vuoti in NaN (serve a Pandas)
# lo 0 è perche molti cartellini non hanno un valore e sono segnati come 0
# Es. tutti i giocatori dell'atalanta hanno 0 cartellini (gialli e rossi), cosa impossibile
df_excel = df_excel.replace(['N/D', 'null', '', 0], pd.NA)

# ID Giocatore come indice del DataFrame
df_excel.set_index('ID Giocatore', inplace=True)

log = []
dati_json = []

# prendo u file alla volta dalla cartella
for filename in os.listdir(folder_json):
    if filename.endswith('.json'):

        # costruisco il filepath per aprire il json
        filepath = os.path.join(folder_json, filename)
        
        with open(filepath, 'r', encoding='utf-8') as file:
            try:
                json_content = json.load(file)
                player_data = json_content.get('pageProps', {}).get('data', {})
                
                if not player_data:
                    log.append(f"Struttura JSON non valida (NO PLAYER DATA) in --> {filename}")
                    continue
                    
                player_id = player_data.get('id')
                player_name = player_data.get('name')
                
                # Inizializzo un dizionario utilizzando l'ID
                player_dict = {
                    'ID Giocatore': player_id
                }
                


                # Estrai le statistiche
                stats = player_data.get('mainLeague', {}).get('stats')
                if stats:
                    for stat in stats:
                        title = stat.get('title')
                        value = stat.get('value')
                        if title:
                            player_dict[title] = value
                else:
                    log.append(f"Nessuna statistica trovata per --> {filename}")
                            
                dati_json.append(player_dict)
                
            except Exception as e:
                print(f"Errore durante la lettura del file: {e} --> {filename}")

In [66]:
for i in log:
    print(i)

print(len(log))

Nessuna statistica trovata per --> alejandro-cichero_1581056_player.json
Nessuna statistica trovata per --> alessandro-plizzari_783635_player.json
Nessuna statistica trovata per --> aljaz-strajnar_1804416_player.json
Nessuna statistica trovata per --> andrea-vailati_1685517_player.json
Nessuna statistica trovata per --> eldin-lolic_1329536_player.json
Nessuna statistica trovata per --> francesco-minicangeli_1681814_player.json
Nessuna statistica trovata per --> francesco-reita_1897331_player.json
Nessuna statistica trovata per --> lorenzo-gori_1783113_player.json
Nessuna statistica trovata per --> marco-ladisa_1741048_player.json
Nessuna statistica trovata per --> mathis-mout_1879542_player.json
Nessuna statistica trovata per --> matteo-grandi_415091_player.json
Nessuna statistica trovata per --> matteo-obleac_1818958_player.json
Nessuna statistica trovata per --> nicolo-ballabio_1888215_player.json
Nessuna statistica trovata per --> omari-forson_1190023_player.json
Nessuna statistica 

In [67]:
dati_json_df = pd.DataFrame(dati_json)
print(dati_json_df.head())

   ID Giocatore  Goals  Assists  Started  Matches  Minutes played  \
0       1802590    0.0      0.0      0.0      2.0             8.0   
1       1425693    0.0      2.0      6.0     18.0           765.0   
2       1192576    4.0      1.0      9.0     24.0           873.0   
3       1681988    0.0      0.0      1.0      5.0           163.0   
4       1581056    NaN      NaN      NaN      NaN             NaN   

   Yellow cards  Red cards  Rating  Clean sheets  Goals conceded  \
0           0.0        0.0     NaN           NaN             NaN   
1           1.0        0.0    6.88           NaN             NaN   
2           0.0        0.0    6.47           NaN             NaN   
3           0.0        0.0    6.96           NaN             NaN   
4           NaN        NaN     NaN           NaN             NaN   

  Saved penalties  
0             NaN  
1             NaN  
2             NaN  
3             NaN  
4             NaN  


In [70]:
#cro DataFrame con i dati dei JSON
df_json = pd.DataFrame(dati_json)

#utilizzo combine_first (merge intelligente)
if not df_json.empty:
    df_json.set_index('ID Giocatore', inplace=True)
    df_finale = df_excel.combine_first(df_json)
else:
    print("Nessun dato valido trovato nei JSON.")
    df_finale = df_excel

# faccio tornare 'ID Giocatore' una colonna normale
df_finale.reset_index(inplace=True)

# con questo sostituisco i Nan, ma sarebbe meglio tenerli così per il training
df_finale = df_finale.fillna('N/D')

df_finale.to_excel(r'.\Datasets\PlayerStats\promoted_Players_Stats.xlsx', index=False)

print("Dati uniti con successo! Ecco un'anteprima:")
display(df_finale)

Dati uniti con successo! Ecco un'anteprima:


,ID Giocatore,codice team,nome team,Nome Giocatore,dateOfBirth,età,role,rating,transferValue,Red cards,Yellow cards,Goals,Assists,Started,Matches,Minutes played,Rating,Clean sheets,Goals conceded,Saved penalties
0,163919,6504,Monza,Pedro Obiang,1992-03-27,34,Midfielder,7.07,293963.0,0.0,3,2.0,1.0,28.0,32.0,2340.0,7.08,N/D,N/D,N/D
1,267374,6504,Monza,Hernani,1994-03-27,32,Midfielder,7.11,954405.0,0.0,3,5.0,1.0,13.0,18.0,1052.0,7.12,N/D,N/D,N/D
2,317156,7881,Venezia,Alfred Duncan,1993-03-10,33,Midfielder,6.92,618607.0,1,1,1.0,1.0,8.0,17.0,718.0,6.92,N/D,N/D,N/D
3,413160,6504,Monza,Andrea Petagna,1995-06-30,30,Attacker,6.92,601784.0,0.0,4,9.0,2.0,10.0,27.0,1113.0,6.92,N/D,N/D,N/D
4,414232,7881,Venezia,Ridgeciano Haps,1993-06-12,33,Midfielder,7.01,283384.0,0.0,6,5.0,3.0,13.0,25.0,1220.0,7.02,N/D,N/D,N/D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,1804416,6504,Monza,Aljaz Strajnar,2007-08-16,18,Keeper,N/D,150000.0,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D
93,1818958,9891,Frosinone,Matteo Obleac,2006-06-23,19,Defender,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D
94,1879542,6504,Monza,Mathis Mout,2007-08-06,18,Defender,N/D,20000.0,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D
95,1888215,6504,Monza,Nicolò Ballabio,2008-03-01,18,Midfielder,N/D,60000.0,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D


### Now join promoted teams players data to the player dataset

In [72]:
player_excel = pd.read_excel(r'.\Datasets\Styled_STATS_giocatori_serie_A.xlsx', engine='openpyxl')

In [73]:
player_excel.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   ID Giocatore       623 non-null    int64 
 1   Codice Team        623 non-null    int64 
 2   Nome Team          623 non-null    str   
 3   Nome Giocatore     623 non-null    str   
 4   Data di Nascita    623 non-null    str   
 5   Età                623 non-null    int64 
 6   Ruolo              623 non-null    str   
 7   Rating             623 non-null    object
 8   Valore di Mercato  623 non-null    object
 9   Red Cards          623 non-null    object
 10  Yellow Cards       623 non-null    object
 11  Goals              623 non-null    object
 12  Assists            623 non-null    object
 13  Started            623 non-null    object
 14  Matches            623 non-null    object
 15  Minutes played     623 non-null    object
 16  Yellow cards       623 non-null    object
 17  Red card

In [74]:
df_finale.info()

<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ID Giocatore     97 non-null     int64 
 1   codice team      97 non-null     str   
 2   nome team        97 non-null     str   
 3   Nome Giocatore   97 non-null     str   
 4   dateOfBirth      97 non-null     str   
 5   età              97 non-null     int64 
 6   role             97 non-null     str   
 7   rating           97 non-null     object
 8   transferValue    97 non-null     object
 9   Red cards        97 non-null     object
 10  Yellow cards     97 non-null     object
 11  Goals            97 non-null     object
 12  Assists          97 non-null     object
 13  Started          97 non-null     object
 14  Matches          97 non-null     object
 15  Minutes played   97 non-null     object
 16  Rating           97 non-null     object
 17  Clean sheets     97 non-null     object
 18  Goa

#### Small Fixes
I have to remove from <b>df_finale</b> column 16 (duplicate) <br>
I have to remove from <b>player_excel</b> columns 9 and 10 (duplicate) <br>
I have to rename each column of <b>df_finale</b>

In [ ]:
df_finale.drop(df_finale.columns[16], axis=1, inplace=True)
player_excel.drop(player_excel.columns[[9, 10]], axis=1, inplace=True)


rename_dict = {
    'codice team': 'Codice Team',
    'nome team': 'Nome Team',
    'dateOfBirth': 'Data di Nascita',
    'età': 'Età',
    'role': 'Ruolo',
    'rating': 'Rating',
    'transferValue': 'Valore di Mercato'
}
df_finale.rename(columns=rename_dict, inplace=True)


df_unito = pd.concat([player_excel, df_finale], ignore_index=True)

#### Style For Readability

In [ ]:
#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Risultati e Statistiche"
ws_data.views.sheetView[0].showGridLines = True


# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)


# Populate Data Sheet
headers = list(df_unito.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df_unito.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)


ws_data.freeze_panes = "A2"
path_stats_styled = r".\Datasets\Styled_PROMOTED_giocatori_serie_A.xlsx"
wb.save(path_stats_styled)

## Old Seasons Players

I have to extract data for the old seasons players, given that i haven't extracted any basic data from the teams JSONs, i have to write a custom script in order to extract all the data i need from the players JSONs.<br>
I have already downloaded all JSONs in the folder "<i>.\Download\PlayerStats\PlayersJSONs_Additional_Old</i>"

In [84]:
folder_json = r'.\Datasets\PlayerStats\PlayersJSONs_Additional_old'
log = []
dati_json = []


for filename in os.listdir(folder_json):
    if filename.endswith('.json'):
        filepath = os.path.join(folder_json, filename)
        
        with open(filepath, 'r', encoding='utf-8') as file:
            try:
                json_content = json.load(file)
                player_data = json_content.get('pageProps', {}).get('data', {})
                
                if not player_data:
                    log.append(f"Struttura JSON non valida (NO PLAYER DATA) in --> {filename}")
                    continue
                
                team_data = player_data.get('team') or player_data.get('primaryTeam') or {}
                
                #prendo data nascita
                data_nascita = player_data.get('dateOfBirth')
                if not data_nascita and player_data.get('birthDate'):
                    data_nascita_completa = player_data.get('birthDate', {}).get('utcTime', '')
                    if data_nascita_completa:
                        data_nascita = data_nascita_completa.split('T')[0] 

                # prendo ruolo
                ruolo = player_data.get('position') or player_data.get('role')
                if not ruolo:
                    pos_desc = player_data.get('positionDescription') or {}
                    ruolo = pos_desc.get('primaryPosition', {}).get('label')

                # prendo market value
                eta = player_data.get('age') or player_data.get('età')
                valore_mercato = player_data.get('transferValue') or player_data.get('marketValue')
                
                # per eta e nel caso anmcora market value
                player_info_list = player_data.get('playerInformation') or []
                for info in player_info_list:
                    title = info.get('title')
                    value_obj = info.get('value') or {}
                    
                    if title == 'Age' and not eta:
                        eta = value_obj.get('numberValue') or value_obj.get('fallback') 
                        
                    elif title == 'Market value' and not valore_mercato:
                        valore_mercato = value_obj.get('fallback') or value_obj.get('numberValue')

                
                player_dict = {
                    'ID Giocatore': player_data.get('id'),
                    'Codice Team': team_data.get('id') or team_data.get('teamId'),
                    'Nome Team': team_data.get('name') or team_data.get('teamName'),
                    'Nome Giocatore': player_data.get('name'),
                    'Data di Nascita': data_nascita,
                    'Età': eta,
                    'Ruolo': ruolo,
                    'Rating': player_data.get('rating'),
                    'Valore di Mercato': valore_mercato
                }
                
                # Estrazione statistich
                main_league = player_data.get('mainLeague') or {}
                stats = main_league.get('stats')
                
                if stats:
                    for stat in stats:
                        title = stat.get('title')
                        value = stat.get('value')
                        if title:
                            if title == 'Red Cards': title = 'Red cards'
                            if title == 'Yellow Cards': title = 'Yellow cards'
                            player_dict[title] = value
                else:
                    log.append(f"Nessuna statistica trovata per --> {filename}")
                        
                dati_json.append(player_dict)
                
            except Exception as e:
                print(f"Errore durante la lettura del file {filename}: {e}")


df_nuovo_json = pd.DataFrame(dati_json)


In [85]:
print(df_nuovo_json.head())

   ID Giocatore  Codice Team        Nome Team         Nome Giocatore  \
0       1436505       6106.0      Juve Stabia  Aaron Ciammaglichella   
1        690516       9906.0  Atletico Madrid        Ademola Lookman   
2        920861       4685.0        Kasımpaşa      Adrian Benedyczak   
3       1075450      10077.0     Boca Juniors      Agustín Martegani   
4        792653      10200.0    Panathinaikos            Ahmed Touba   

  Data di Nascita  Età        Ruolo    Rating Valore di Mercato  Goals  \
0      2005-01-26   21   midfielder  6.510000           €740.5k    0.0   
1      1997-10-20   28      Striker  6.770000            €40.2m    4.0   
2      2000-11-24   25      Striker  7.340000             €1.9m   10.0   
3      2000-05-20   26   midfielder       NaN             €689k    NaN   
4      1998-03-13   28  Center Back  6.907273             €1.6m    0.0   

   Assists  Started  Matches  Minutes played  Yellow cards  Red cards  \
0      0.0      1.0      2.0            70.0     

In [90]:
last_df = pd.concat([df_unito, df_nuovo_json], ignore_index=True)
last_df = last_df.fillna('N/D')

#### Style for Readability

In [ ]:
#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Risultati e Statistiche"
ws_data.views.sheetView[0].showGridLines = True


# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)


# Populate Data Sheet
headers = list(last_df.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in last_df.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)


ws_data.freeze_panes = "A2"
path_stats_styled = r".\Datasets\Styled_Players_Final.xlsx"
wb.save(path_stats_styled)

## Fix Player Rating
The player dataset contains a lot of players which hasn't played a single game or hasn't played long enough to get a rating. <br> We can fix this issue with three solutions:
1. Assign the average rating of their whole team to these players
2. Assign the lowest rating in their team to these players
3. Count the reating of these players as zero

Solution evaluation:
- Giving the average rating is too good, these players hasn't played a single game because they are not better than the rest of the team
- Giving zero as a rating is too much penalizing, these players are not that bad, they are just not good enougth to play for a consistent amount of time
- Giving the lowest rating is the best solution because is not as penalizing as giving 0 and it is not as rewarding as giving the team average

In [92]:
df_rating = last_df

# Rating in numeric, se non si può lo rende NaN
df_rating['Rating'] = pd.to_numeric(df_rating['Rating'], errors='coerce')

# creo vettore contenente il rating minimo delle squadre 
rating_minimo_per_squadra = df_rating.groupby('Nome Team')['Rating'].transform('min')

# cambio i valori NaN con i valori del vettore
df_rating['Rating'] = df_rating['Rating'].fillna(rating_minimo_per_squadra)

# Se una squadra ha solo Nan metto il rating a 0 (per dati dei vecchi giocatori che ora sono in altri team)
df_rating['Rating'] = df_rating['Rating'].fillna(0)

#### Style for Readibility

In [ ]:
#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Risultati e Statistiche"
ws_data.views.sheetView[0].showGridLines = True


# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)


# Populate Data Sheet
headers = list(df_rating.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df_rating.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)


ws_data.freeze_panes = "A2"
path_stats_styled = r".\Datasets\Styled_Players_Final.xlsx"
wb.save(path_stats_styled)

## Creation of Team Rating columns in Fixture Datasets

1. Read ID Giocatore and Rating from Player Dataset
2. Read Fixture Dataset
3. For each match extract player list for Home and Away team
4. Calculate the mean value for the teams rating matching the players ID

In [96]:
players_rating = pd.read_excel(r".\Datasets\Styled_Players_Final.xlsx", engine='openpyxl')
players_rating = players_rating[["ID Giocatore", "Rating"]]
players_rating.info()

<class 'pandas.DataFrame'>
RangeIndex: 1219 entries, 0 to 1218
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID Giocatore  1219 non-null   int64  
 1   Rating        1219 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 19.2 KB


In [97]:
fixture_dataset = pd.read_excel(r".\Datasets\Styled_Fixtures_Lineup.xlsx", engine='openpyxl')
fixture_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 97 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Stagione                         1140 non-null   str    
 1   Giornata                         1140 non-null   int64  
 2   ID Partita                       1140 non-null   int64  
 3   Data (UTC)                       1140 non-null   str    
 4   Squadra Casa                     1140 non-null   str    
 5   Squadra Fuori                    1140 non-null   str    
 6   Risultato                        1140 non-null   str    
 7   Gol Casa                         1140 non-null   int64  
 8   Gol Fuori                        1140 non-null   int64  
 9   Stato                            1140 non-null   str    
 10  URL FotMob                       1140 non-null   str    
 11  Ball possession_Casa             1140 non-null   int64  
 12  Ball possession_Fuori          

In [99]:
print(fixture_dataset.iloc[1]["HomeStartingTeam"])

Stefano Turati_869054, Anthony Oyono_1235421, Ilario Monterisi_1117058, Simone Romagnoli_111739, Riccardo Marchizza_719946, Francesco Gelli_618698, Luca Mazzitelli_513982, Abdou Harroui_909030, Jaime Báez_391991, Marvin Cuni_956212, Giuseppe Caso_979789


In [100]:

for index, row in fixture_dataset.iterrows():
    home_starting = row["HomeStartingTeam"]

    if(home_starting != "N/D"):
        print(home_starting)
        print(type(home_starting))
        break

Elia Caprile_958422, Tyronne Ebuehi_561844, Ardian Ismajli_750880, Sebastiano Luperto_604330, Liberato Cacace_919671, Alberto Grassi_575364, Razvan Marin_574844, Matteo Cancellieri_1146500, Tommaso Baldanzi_1200591, Emmanuel Quartsin Gyasi_526450, Francesco Caputo_155254
<class 'str'>


In [107]:
def getRating_by_ID(player_ID):
    
    try:
        player_ID_num = int(player_ID)
    except ValueError:
        print(f"Errore su player_ID: {player_ID} non è un numero intero valido.")
        return 0 

    risultato = players_rating.loc[players_rating['ID Giocatore'] == player_ID_num, 'Rating']
    
    if not risultato.empty:
        return risultato.values[0]
    else:
        print(f"Attenzione: Giocatore con ID {player_ID_num} non trovato nel dataset. Assegnato 0.")

In [105]:
# esempio con Darmian
print(getRating_by_ID(46820))

6.37


In [111]:
# in poche occasioni il valore potrebbe essere NaN, quidni lo rendo ""
fixture_dataset['HomeStartingTeam'] = fixture_dataset['HomeStartingTeam'].fillna("")
fixture_dataset['HomeSubPlayers'] = fixture_dataset['HomeSubPlayers'].fillna("")
fixture_dataset['AwayStartingTeam'] = fixture_dataset['AwayStartingTeam'].fillna("")
fixture_dataset['AwaySubPlayers'] = fixture_dataset['AwaySubPlayers'].fillna("")

In [ ]:
copy_df = fixture_dataset.copy()
home_ratings = []
away_ratings = []

for index, row in fixture_dataset.iterrows():
    home_starting = row["HomeStartingTeam"]
    home_sub = row["HomeSubPlayers"]
    away_starting = row["AwayStartingTeam"]
    away_sub = row["AwaySubPlayers"]

    home_team_str = home_starting + ", " + home_sub
    away_team_str = away_starting + ", " + away_sub


    # faccio per home team
    counter = 0
    rating_sum = 0

    for player in home_team_str.split(","):
        if not player.strip(): 
            continue
            
        playerID = player.split("_")[-1].strip()
        rating_sum = rating_sum + getRating_by_ID(playerID)
        counter = counter + 1
        
    # Evita l'errore di divisione per zero se una squadra risulta vuota
    home_team_rating = (rating_sum / counter) if counter > 0 else 0


    # faccio per away team
    counter = 0
    rating_sum = 0

    for player in away_team_str.split(","):
        if not player.strip(): 
            continue
            
        playerID = player.split("_")[-1].strip()
        rating_sum = rating_sum + getRating_by_ID(playerID)
        counter = counter + 1

    # Evita l'errore di divisione per zero se una squadra risulta vuota
    away_team_rating = (rating_sum / counter) if counter > 0 else 0

    home_ratings.append(home_team_rating)
    away_ratings.append(away_team_rating)

copy_df = copy_df.assign(
    HomeTeamRating = home_ratings,
    AwayTeamRating = away_ratings
)


In [113]:
copy_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 99 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Stagione                         1140 non-null   str    
 1   Giornata                         1140 non-null   int64  
 2   ID Partita                       1140 non-null   int64  
 3   Data (UTC)                       1140 non-null   str    
 4   Squadra Casa                     1140 non-null   str    
 5   Squadra Fuori                    1140 non-null   str    
 6   Risultato                        1140 non-null   str    
 7   Gol Casa                         1140 non-null   int64  
 8   Gol Fuori                        1140 non-null   int64  
 9   Stato                            1140 non-null   str    
 10  URL FotMob                       1140 non-null   str    
 11  Ball possession_Casa             1140 non-null   int64  
 12  Ball possession_Fuori          

#### Style for Readability

In [ ]:
#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Risultati e Statistiche"
ws_data.views.sheetView[0].showGridLines = True


# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)


# Populate Data Sheet
headers = list(copy_df.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in copy_df.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    # Apply link to URL FotMob column
    url_val = r_data["URL FotMob"]
    if url_val:
        cell = ws_data.cell(row=row_num, column=headers.index("URL FotMob") + 1)
        cell.hyperlink = url_val
        cell.font = Font(name="Segoe UI", size=10, color="0000FF", underline="single")
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            if col_idx != headers.index("URL FotMob") + 1:
                c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)


ws_data.freeze_panes = "A2"
path_stats_styled = r".\Datasets\Styled_Fixture_Final.xlsx"
wb.save(path_stats_styled)